<a href="https://colab.research.google.com/github/legna7816/ml-projects/blob/main/nlp_sentiment/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NLP 감정분석 프로젝트
# 흐름: 텍스트 -> 숫자 변환 -> IMDB 실전 감정분석 -> 워드 임베딩

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# 1. 텍스트를숫자로 바꾸는 방법 (BoW -> TF-IDF)
# 모델은 숫자만 이해 -> 텍스트도 숫자 벡터로 변환해야 함

corpus = [
    "the food was delicious",
    "the food was terrible",
    "i will visit again"
]

# 1-1. BoW (Bag of Words): 단어 등장 횟수로 벡터화
bow = CountVectorizer()
X_bow = bow.fit_transform(corpus) # fit: 단어 사전 생성, transform: 벡터 변환

print('1-1')
print(bow.get_feature_names_out()) # 전체 단어 사전
print(X_bow.toarray()) # 행=문장, 열=단어, 값=등장 횟수

# 하지만 "delicious"와 "terrible"로 문장 의미가 정반대인데,
# 나머지 단어(the/food/was)가 같아서 두 벡터가 매우 비슷하게 보임

# 1-2. TF-IDF: 흔한 단어는 가중치 낮게, 특징적 단어는 높게
tfidf_demo = TfidfVectorizer()
X_tfidf_demo = tfidf_demo.fit_transform(corpus)

print('\n1-2')
print(tfidf_demo.get_feature_names_out())
print(X_tfidf_demo.toarray().round(2))

# 모든 문장에 나오는 the/food/was는 0.46
# delicious/terrivle은 0.60으로 더 높은 값 -> 중요 단어가 자동으로 부각됨

# 1-3. stop_words: 의미 없는 흔한 단어(the, was, i...) 자동 제거
tfidf_sw = TfidfVectorizer(stop_words='english')
X_tfidf_sw = tfidf_sw.fit_transform(corpus)

print('\n1-3')
print(tfidf_sw.get_feature_names_out())
# ['delicious', 'food', 'terrible', 'visit']만 남음
# 감정 분석에는 'not'도 stop word로 제거되기에 "not good" 같은 부정 표현을 놓칠 수 있음

1-1
['again' 'delicious' 'food' 'terrible' 'the' 'visit' 'was' 'will']
[[0 1 1 0 1 0 1 0]
 [0 0 1 1 1 0 1 0]
 [1 0 0 0 0 1 0 1]]

1-2
['again' 'delicious' 'food' 'terrible' 'the' 'visit' 'was' 'will']
[[0.   0.6  0.46 0.   0.46 0.   0.46 0.  ]
 [0.   0.   0.46 0.6  0.46 0.   0.46 0.  ]
 [0.58 0.   0.   0.   0.   0.58 0.   0.58]]

1-3
['delicious' 'food' 'terrible' 'visit']


In [3]:
# 2. 실전 감정분석 - IMDB 영화 리뷰 (긍정/부정 각 25,000개)

# 2-1. 데이터 다운로드
if not os.path.exists('aclImdb'):
  os.system('wget -q http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz')
  os.system('tar -xzf aclImdb_v1.tar.gz')

# 2-2. 파일에서 리뷰 텍스트 읽기
def load_imdb(path, split='train'):
  """net/pos 폴더의 리뷰 파일을 읽어 (텍스트 리스트, 라벨 리스트) 반환. neg=0, pod=1"""
  texts, labels = [], []
  for label, sentiment in enumerate(['neg', 'pos']):
    folder = os.path.join(path, split, sentiment)
    for fname in os.listdir(folder):
      with open(os.path.join(folder, fname), encoding='utf-8') as f:
        texts.append(f.read())
      labels.append(label)
  return texts, labels

train_texts, train_labels = load_imdb('aclImdb', 'train')
df_train = pd.DataFrame({'text': train_texts, 'label': train_labels})

print('2-1')
print(df_train.shape) # (25000, 2)
print(df_train['label'].value_counts()) # 긍정/부정 12500개씩 균형 확인

# 2-3. EDA: 리뷰 길이 분포 확인
df_train['review_len'] = df_train['text'].apply(lambda x: len(x.split()))
print('\n2-2')
print(df_train['review_len'].describe()) # 평균/최소/최대 단어 수 파악

# 2-4. 샘플링 (전체 25000개는 느리므로 10000개로 실습)
df_sample = df_train.sample(10000, random_state=42)
X_text = df_sample['text'].values
y = df_sample['label'].values

# 2-5. TF-IDF 벡터화
# max_feature=5000: 상위 5000개 단어만 사용 (차원 제한 -> 속도/성능 균형)
# ngram_range=(1, 2): 단어 1개(unigram) + 2개 조합(bigram) 모두 특징으로 사용, bigram이 "not good" 같은 부정 표현을 잡는 데 유리
tfidf = TfidfVectorizer(stop_words='english',
                        max_features=5000,
                        ngram_range=(1, 2))
X = tfidf.fit_transform(X_text)
print('\n2-5')
print('벡터 크기: ', X.shape)

# 2-6. 모델 학습 평가
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 모델 1: LogisticRegression
# 실험 결과: 데이터가 늘수록 성능 상승 (5000개 84.5% -> 10000개 86.4%)
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
log_pred = log_model.predict(X_test)
print('\n2-6')
print('Logistic 정확도: ', accuracy_score(y_test, log_pred))
print(classification_report(y_test, log_pred, target_names=['부정', '긍정']))

# 모델 2: MultinomialNB(나이브베이즈)
# "이 단어들이 나왔을 떄 긍정/부정일 확률"을 계산하는 방식
# 실험 결과: 데이터가 적어도 잘 작동하지만, 늘려도 크게 안 오름 (84.3% -> 84.2%)
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
print('NaiveBayes 정확도: ', accuracy_score(y_test, nb_pred))

# 2-7. 모델 해석; 어떤 단어가 긍정/부정을 가르는가
# LogisticRegression의 가중치 (coefficient) = 각 단어의 긍정/부정 영향력
feature_names = tfidf.get_feature_names_out()
coefs = log_model.coef_[0]

print('\n2-7')
print('긍정 키워드 top10')
print(pd.Series(coefs, index=feature_names).nlargest(10))
print('부정 키워드 top10')
print(pd.Series(coefs, index=feature_names).nsmallest(10))
# 실험 결과: worst, bad, boring, awful 등이 부정 상위
# "not good" 같은 bigram은 없음 -> max_features=5000 상한에 못 들어왔고,
# stop_words='english'가 'not'을 제거한 영향도 있음

# 2-8. 새 리뷰 직접 예측
my_reviews = [
    "this movie was absolutely fantastic and the acting was superb",
    "terrible film boring and waste of time awful experience"
]
# 새 데이터에는 transform만 사용 (fit_transform 금지)
# fit_transform을 쓰면 학습 때 만든 단어 사전/가중치가 새로 덮어써져서
# 모델이 학습한 기준과 달라짐 -> 데이터 누구 (data leakage) 방지 원칙
X_my = tfidf.transform(my_reviews)
print('\n2-8')
print('내 리뷰 예측 (1: 긍정, 0: 부정): ', log_model.predict(X_my))

2-1
(25000, 2)
label
0    12500
1    12500
Name: count, dtype: int64

2-2
count    25000.000000
mean       233.787200
std        173.733032
min         10.000000
25%        127.000000
50%        174.000000
75%        284.000000
max       2470.000000
Name: review_len, dtype: float64

2-5
벡터 크기:  (10000, 5000)

2-6
Logistic 정확도:  0.8635
              precision    recall  f1-score   support

          부정       0.87      0.85      0.86       971
          긍정       0.86      0.87      0.87      1029

    accuracy                           0.86      2000
   macro avg       0.86      0.86      0.86      2000
weighted avg       0.86      0.86      0.86      2000

NaiveBayes 정확도:  0.852

2-7
긍정 키워드 top10
great        4.944694
excellent    3.896263
perfect      3.391824
best         3.258264
wonderful    3.163169
favorite     2.774288
loved        2.732259
beautiful    2.633581
amazing      2.597735
enjoyed      2.592639
dtype: float64
부정 키워드 top10
worst      -6.101393
bad        -5.587408
borin

In [4]:
# 3. 워드 임베딩 - 단어를 "의미를 가진 벡터로"
# TF-IDF의 한계: good과 great가 비슷한 뜻인 걸 모음 (등장 횟수만 봄)
# 임베딩: 단어를 100차원 실수 벡터로 표현, 비슷한 뜻 = 가까운 벡터

# !pip install gensim    # gensim 미설치 시 실행 후 런타임 재시작
import gensim.downloader as api

# 위키피디아로 사전학습된 100차원 Glove 임베딩 (약 128MB, 최소 1회 다운로드)
wv = api.load('glove-wiki-gigaword-100')

# 3.1 단어 -> 벡터 확인
print('3-1')
print(wv['king'][:10])    # 100차원 실수 벡터의 앞 10개 값
print(wv['king'].shape)   # (100, )

# 3-2. 의미적 유사도 (TF-IDF로는 불가능했던 것)
print('\n3-2')
print(wv.most_similar('good', topn=5))    # good과 비슷한 단어 5개
print(wv.similarity('good', 'great'))     # 높음 (유의어)
print(wv.similarity('good', 'terrible'))  # 낮음 (반의어)

# 3-3. 의미 연산: 벡터 공간이 "관계"를 학습했다는 증거
print('\n3-3')
# king - man + woman = queen (성별 관계)
print(wv.most_similar(positive=['king', 'woman'], negative=['man'], topn=3))
# paris - france + italy = rome (수도 관계, 실형 결과 rome이 0.82로 1위)
print(wv.most_similar(positive=['paris', 'italy'], negative=['france'], topn=3))

# 3-4. 정적 임베딩의 한계 발견: 반의어 구분 실패
print('\n3-4')
print('happy vs joyful:', wv.similarity('happy', 'joyful')) # 0.53 (유의어인데 낮음)
print('happy vs sad:', wv.similarity('happy', 'sad'))       # 0.68 (반의어인데 더 높음)
print(wv.most_similar('bad', topn=5))                       # good이 2위
# 이유: good/bad, happy/sad는 같은 문맥("I feel ____")에서 쓰이는데,
#       임베딩은 "주변 단어가 비슷하면 비슷한 벡터"로 학습하기 때문
# 이 한계를 극복하기 위해 나온 것이 문맥 기반 임베딩 (BERT 등 Transformer)

3-1
[-0.32307 -0.87616  0.21977  0.25268  0.22976  0.7388  -0.37954 -0.35307
 -0.84369 -1.1113 ]
(100,)

3-2
[('better', 0.893191397190094), ('sure', 0.8314563035964966), ('really', 0.8297762274742126), ('kind', 0.8288268446922302), ('very', 0.8260800242424011)]
0.7592797
0.5365428

3-3
[('queen', 0.7698540687561035), ('monarch', 0.6843381524085999), ('throne', 0.6755736470222473)]
[('rome', 0.8189547061920166), ('milan', 0.7376196980476379), ('naples', 0.7117615342140198)]

3-4
happy vs joyful: 0.52599776
happy vs sad: 0.6801137
[('worse', 0.7929712533950806), ('good', 0.7702797651290894), ('things', 0.7653602957725525), ('too', 0.7630148530006409), ('thing', 0.7609668374061584)]


In [5]:
# 텍스트 표현 방법의 발전 과정
# 1. BoW/TF-IDF: 단어 의미를 모름 (등장 횟수만)
# 2. 정적 임베딩 : 의미는 알지만 반의어 헷갈림 (Glove)
# 3. 문맥 임베딩: 문맥까지 이해 (BERT, Transformer)